# Solution A2 · Leave-one-out for all k with a single distance matrix

**Type**: Implement from scratch

## Objective

Compute the leave-one-out accuracy of a KNN for **all the `k` from 1 to 60 in a single pass**: an n×n distance matrix with the diagonal excluded, a sort per row and the cumulated votes. Check
the exact equality with `cross_val_score(..., cv=LeaveOneOut())` and measure how much you save.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
Z = StandardScaler().fit_transform(X)
print(Z.shape, "| classes:", np.bincount(y))

## Commented solution

In [ ]:
def loo_matrix(Z):
    sq = (Z ** 2).sum(1)
    D = np.sqrt(np.maximum(sq[:, None] + sq[None, :] - 2 * Z @ Z.T, 0))
    np.fill_diagonal(D, np.inf)                                # no point is its own neighbor
    return D


def loo_accuracy(Z, y, k_max):
    D = loo_matrix(Z)
    order = np.argsort(D, axis=1)[:, :k_max]                   # the k_max neighbors of every point, from the nearest
    oh = np.eye(y.max() + 1)[y[order]]                         # (n, k_max, classes): the class of each neighbor
    votes = np.cumsum(oh, axis=1)                              # votes[i, k-1, c] = votes for c among the first k neighbors of i
    return (votes.argmax(2) == y[:, None]).mean(0)             # accuracy for k = 1..k_max

In [ ]:
t0 = time.time()
slow = {k: cross_val_score(KNeighborsClassifier(k), Z, y, cv=LeaveOneOut()).mean() for k in (1, 4, 7, 25)}
t_slow = time.time() - t0
t0 = time.time()
curve = loo_accuracy(Z, y, 60)
t_fast = time.time() - t0
speedup = t_slow / t_fast
print(f"scikit-learn (4 values of k): {t_slow:.1f}s | yours (60 values of k): {t_fast:.3f}s | ratio: {speedup:.0f}x")

In [ ]:
best_k = int(np.argmax(curve)) + 1
simple_rule_k = max(k for k in range(1, 61) if curve[k - 1] >= curve.max() - 0.01)
print(f"LOO accuracy for k = 1, 5, 15, 30, 60: {curve[[0, 4, 14, 29, 59]].round(4)}")
print(f"best k: {best_k} (accuracy {curve.max():.4f}) | k of the simple rule (within 1 point): {simple_rule_k}")

## Check

In [ ]:
with verify():
    D = loo_matrix(Z)
    check(D.shape == (569, 569) and np.isinf(np.diag(D)).all() and np.isfinite(D[~np.eye(569, dtype=bool)]).all(), "569 x 569 matrix with the diagonal at infinity", "np.fill_diagonal(D, np.inf)")
    check(len(curve) == 60 and all(abs(curve[k - 1] - slow[k]) < 1e-12 for k in (1, 4, 7, 25)), "the LOO accuracy matches scikit-learn exactly for k = 1, 4, 7, 25", "votes among the first k neighbors, tie -> smallest label")
    check(speedup > 5, "your function is over 5 times faster (60 values of k against 4)", "a single sort of the rows")
    check(best_k == int(np.argmax(curve)) + 1, "best k: the one with maximum accuracy (on a tie, the smallest)", "np.argmax(curve) + 1")
    check(simple_rule_k == max(k for k in range(1, 61) if curve[k - 1] >= curve.max() - 0.01) and simple_rule_k >= best_k, "k of the simple rule: the largest within one point of the maximum", "smoother model at equal performance")
done()

## What you learned

- With a single sort per row you get **the whole curve** of the accuracy as a function of `k`: the predictions for `k` are the cumulated votes of the first `k` neighbors, and the votes for `k+1` are obtained from those for `k` by adding one neighbor.
  It is a property specific to KNN (parametric models have nothing like it) and explains why LOO is cheap for KNN.
- The curve is **flat** around the maximum: many `k` are equivalent within the uncertainty of the estimate (about ±1 point with 569 examples). Choosing the argmax is choosing the noise; the rule "the largest `k` within one point
  of the maximum" (smoother model) is more robust.
- LOO has a single notable cost: the n×n matrix (569² = 324,000 numbers here, but 10⁸ with 10,000 rows). With larger datasets you move to a K-fold or work in blocks.
- Note about a small sin: the scaler was learned on all the data. For choosing `k` the bias is negligible; for estimating the final performance it is not.

## Common pitfall

Choosing `k` by looking at the single maximum of the LOO curve: with 569 examples the difference between `k = 7` and `k = 15` is less than the noise of the estimate. Look at the shape of the curve and choose the simplest model among the equivalent ones.

## Interview question

> *Click on a question to show its answer.*

<details><summary>Why is leave-one-out cross-validation cheap for KNN?</summary>

Because KNN has no training: the prediction for a point uses its neighbors in the training set excluding the point itself, so a single distance matrix with the diagonal excluded is enough. With a single sort per row you get the
predictions for every `k` at almost no cost. What remains is the O(n²) cost of the matrix, and the variance of the LOO estimate (the n estimates are highly correlated), which is why on large datasets I prefer a repeated K-fold.

</details>